# Step: 1 PyPDF loader to load the pdf file

In [1]:
# The below dependency is used to load the pdf file 
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loader = PyPDFLoader("refrigerators.pdf")

documents = loader.load()

print("Total pages:", len(documents))

print(documents[0].page_content)

C:\Users\Dell\AppData\Local\Temp\ipykernel_15712\1266572063.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages: 6
ROHINI COLLEGE OF ENGINEERING AND TECHNOLOGY 
 
                                                                                                                                                    PH8151 ENGINEERING PHYSICS 
 
 
 3.7 Refrigerators 
Refrigeration is the process of continuous cooling or extraction of heat to below that of the 
atmosphere from a substance with a help of the external work. It is based on the second law of 
thermodynamics that the heat can made to flow from cold body to a hot body with the help of 
external source. 
Refrigerator is a machine which produces cold. It is used to remove heat from  the  
refrigerated space and reject it to atmosphere. Hence it maintains the temperature below the 
surrounding atmosphere. The working fluid used in refrigerator is called refrigerant. Ex: 
Ammonia, Methyl chloride, Freon etc. 
Parts of a Refrigerator and their functions 
Evaporator: It is used to re ceive the liquid refrigerants from the condenser thro

In [2]:
# # To print the entire pdf file 

# for doc in documents:
#     print(f"--- Page {doc.metadata['page'] + 1} ---")
#     print(doc.page_content)

# step 2: Split documents into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"], #["\n\n"] This will perform paragraph wise splitting. ["\n"] represents a new line character. 
    chunk_size=500, # Each chunk contains approximately 500 characters.
    chunk_overlap=50 # Chunk overlap means repeating some text from the previous chunk in the next chunk.
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

for chunk in chunks:
    print(chunk.page_content)
    print("-------------------")

Total chunks: 22
ROHINI COLLEGE OF ENGINEERING AND TECHNOLOGY 
 
                                                                                                                                                    PH8151 ENGINEERING PHYSICS 
 
 
 3.7 Refrigerators 
Refrigeration is the process of continuous cooling or extraction of heat to below that of the 
atmosphere from a substance with a help of the external work. It is based on the second law of
-------------------
thermodynamics that the heat can made to flow from cold body to a hot body with the help of 
external source. 
Refrigerator is a machine which produces cold. It is used to remove heat from  the  
refrigerated space and reject it to atmosphere. Hence it maintains the temperature below the 
surrounding atmosphere. The working fluid used in refrigerator is called refrigerant. Ex: 
Ammonia, Methyl chloride, Freon etc. 
Parts of a Refrigerator and their functions
-------------------
Parts of a Refrigerator and their function

Step 3 — Embedding Model to convert text into vectors

In [4]:
# Here we are using huggingFace model all-MiniLM-L6-v2. This is used to convert the text into vectors/Numerical values. all-MiniLM-L6-v2 convert it into 384 dimensions
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("HuggingFace embedding model initialized")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFace embedding model initialized


# Step 4 — Store embeddings in ChromaDB

In [5]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Documents stored successfully!")

Documents stored successfully!


In [9]:
## This below code will show how data is stored in chromaDB
# from langchain_chroma import Chroma
# import json

# vector_store = Chroma.from_documents(
#     documents=chunks,
#     embedding=embeddings,
#     persist_directory="./chroma_db"
# )

# print("Documents stored successfully!")

# # Retrieve stored data from ChromaDB
# stored_data = vector_store.get(
#     include=["documents", "metadatas", "embeddings"]
# )

# # Print IDs
# print("\n========== DOCUMENT IDs ==========")
# print(json.dumps(stored_data["ids"], indent=4))

# # Print original document content
# print("\n========== DOCUMENTS ==========")
# print(json.dumps(stored_data["documents"], indent=4))

# # Print metadata
# print("\n========== METADATA ==========")
# print(json.dumps(stored_data["metadatas"], indent=4))

# # Print embedding vectors
# print("\n========== EMBEDDINGS ==========")
# for i, embedding in enumerate(stored_data["embeddings"]):
#     print(f"\nDocument {i + 1}")
#     print("Embedding dimensions:", len(embedding))
#     print("Embedding vector:", embedding)

# Step 5 — Create a retriever

In [ ]:
# Dense Vector Retrieval (Semantic Search)
# What does k=3 mean?
# When a student asks a question, the retriever finds the three most relevant chunks from the vector database.
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
##-- Here we are checking how our retrival is working --##

# question = "Vapour Absorption Refrigeration Principle"

# results = retriever.invoke(question)

# for result in results:
#     print(result.page_content)
#     print("-------------------")

# Step 6 — Connect the LLM(DeepSeek)

In [24]:
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-flash", # deepseek-flash , deepseek-chat  
    temperature=0
)

print("LLM initialized")

LLM initialized


In [28]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.
user will ask question related to his document try to answer that
u will get the chunks from the vector database refer them before answering the answer should be clear in depth explainatopn should be there
If no chunks is retrived then "Politily mention plz rephase ur questions"
"

Context:
{context}

Question:
{question}

Answer:
""")

# Step 7 — Generate the final answer

- Retriever: Retrieves relevant information from ChromaDB.

- Prompt Template: Combines retrieved information with the user's question.

- LLM: Reads the context and generates a human-readable answer.

In [ ]:
# 1: User asks a question

question = "can you provide me the overall summary of this document?" # Vapour Absorption Refrigeration Principle

# 2: Retrieve relevant documents from ChromaDB
# Retriever searches the vector database and returns
# the top K relevant document chunks.

retrieved_docs = retriever.invoke(question)

# 3: Combine all retrieved document chunks
# Each document contains page_content.
# We use join() to combine multiple chunks into one string.
# \n\n adds two new lines between each chunk.

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

# 4: Prepare the prompt
# Pass the retrieved context and user question
# into the prompt template.
#
# {context}  -> Retrieved information
# {question} -> User's original question

formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

# 5: Send the formatted prompt to the LLM   
# The LLM reads the context and question
# and generates an answer based on the provided information.
response = llm.invoke(formatted_prompt)

# 6: Display the generated answer
# response.content contains the actual text generated by the LLM.
print("AI Answer:")
print(response.content)

AI Answer:
Based on the retrieved chunks, the document is from **Rohini College of Engineering and Technology, PH8151 Engineering Physics**, and it discusses the **principle of refrigeration**, specifically around **Fig. 3.7.3 – Refrigeration principle**.

### Overall summary of the retrieved content
The excerpt mainly describes the **construction** of a refrigeration system. It states that:

- A **throttle valve** is connected between an **evaporator** and a **condenser**.
- A **pump** is connected between an **absorber** and a **separator**.
- **Water is filled in the absorber**.

So, the document appears to introduce the basic components and arrangement of a refrigeration system, likely as part of a larger explanation of the refrigeration cycle/principle. However, the retrieved text is repeated and cut off after “Water is filled in the absorber and it is…”, so the complete working process, cycle explanation, and remaining construction details are not available in the provided chunks